In [18]:
import torch
import torch.nn.functional as F

In [19]:
def get_data(n_users, n_items):
    A = torch.randint(0, 5, (n_users, n_items), dtype=torch.float)
    A[A < 2] = 0 # to make it sparse
    return A

In [20]:
n_users = 15
n_items = 7

In [21]:
data = get_data(n_users, n_items)

In [ ]:
def pearson_correaltion(A):
    rated = (A != 0).float() #mask weather rated or not rated
    # denominator making sure we only take rated entries into considration
    user_means = (A * rated).sum(dim=1) / rated.sum(dim=1).clamp(min=1)
    centered_A = A - user_means.unsqueeze(1)
    centered_A *= rated #reset unrated entries to 0

    # compute the covariance
    numerator = centered_A @ centered_A.T
    row_norms = torch.norm(centered_A, p=2, dim=1, keepdim=True)
    denominator = row_norms @ row_norms.T
    denominator = denominator.clamp(min=1e-6)
    similarity_matrix = numerator / denominator
    return torch.clamp(similarity_matrix, -1.0, 1.0)

In [34]:
similarity_matrix = pearson_correaltion(data)

similarity_matrix.shape


torch.Size([15, 1])


torch.Size([15, 15])

In [24]:
def find_k(similarity_matrix, user_id, k):
    scores = similarity_matrix[user_id]
    scores[user_id] = -1 # exclude itself

    return torch.topk(scores, k).indices


In [25]:
def recommend_items(rating, top_k, target):
    result = {}
    items_to_consider = torch.where(rating[target] == 0)[0] #items not yet rated by target user
    target_user_mean = rating[target].mean()

    for idx in items_to_consider:
        numerator_sum  = 0
        denominator_sum = 0

        for neighbor_idx in top_k:
            #get ratings for this item from neighbours
            neighbor_rating =  rating[neighbor_idx, idx]
            if (neighbor_rating != 0):
                neighbor_mean = rating[neighbor_idx][rating[neighbor_idx] != 0].mean()
                similarity_score = similarity_matrix[target, neighbor_idx]
                numerator_sum += similarity_score * (neighbor_rating - neighbor_mean)
                denominator_sum += abs(similarity_score)

        if (denominator_sum > 0):
            predicted_rating = target_user_mean + (numerator_sum / denominator_sum)
            result[idx.item()] = predicted_rating.item()


    return sorted(result.items(), key=lambda x: x[1], reverse=True)


In [26]:
target = 10
top_k = find_k(similarity_matrix, target, 5)

recommend_items(data, top_k, target)

[(6, 2.7076945304870605), (2, 2.283440113067627), (0, 1.9206827878952026)]